<!-- parity-note -->
## MATLAB Parity Note
- Source MATLAB helpfile: `StimulusDecode2D.mlx`
- Fidelity status: `partial`
- Remaining justified differences: The notebook reproduces the MATLAB section order, figure inventory, simulated receptive fields, and decoded-trajectory presentation, but the current Python decoder still uses regression-based state recovery instead of MATLAB's symbolic-CIF nonlinear filter.


In [ ]:
# nSTAT-python notebook example: StimulusDecode2D
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
SRC_PATH = (REPO_ROOT / "src").resolve()
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from nstat import DecodingAlgorithms
from nstat.notebook_figures import FigureTracker

np.random.seed(0)
OUTPUT_ROOT = REPO_ROOT / "output" / "notebook_images"
__tracker = FigureTracker(topic='StimulusDecode2D', output_root=OUTPUT_ROOT, expected_count=6)


def _prepare_figure(matlab_line: str, *, figsize=(8.0, 4.5)):
    fig = __tracker.new_figure(matlab_line)
    fig.clear()
    fig.set_size_inches(*figsize)
    return fig


def _simulate_decode(seed=19, *, n_cells=24, dt=0.01, tmax=20.0):
    rng = np.random.default_rng(seed)
    time = np.arange(0.0, tmax + dt, dt)
    vel = np.cumsum(rng.normal(0.0, 0.05, size=(time.size, 2)), axis=0)
    vel = 0.18 * vel / np.maximum(np.std(vel, axis=0, ddof=1), 1e-6)
    pos = np.cumsum(vel, axis=0) * dt
    pos = pos - np.mean(pos, axis=0, keepdims=True)
    px = pos[:, 0]
    py = pos[:, 1]
    coeffs = np.column_stack(
        [
            -2.2 - np.abs(rng.normal(0.0, 0.35, size=n_cells)),
            rng.normal(0.0, 1.1, size=n_cells),
            rng.normal(0.0, 1.1, size=n_cells),
            -np.abs(rng.normal(1.6, 0.35, size=n_cells)),
            -np.abs(rng.normal(1.6, 0.35, size=n_cells)),
            rng.normal(0.0, 0.45, size=n_cells),
        ]
    )
    design = np.column_stack([np.ones(time.size), px, py, px * px, py * py, px * py])
    spikes = np.zeros((time.size, n_cells), dtype=float)
    firing_prob = np.zeros_like(spikes)
    for idx in range(n_cells):
        eta = design @ coeffs[idx]
        p = 1.0 / (1.0 + np.exp(-np.clip(eta, -20.0, 20.0)))
        firing_prob[:, idx] = p
        spikes[:, idx] = (rng.random(time.size) < p).astype(float)
    grid = np.linspace(-1.4, 1.4, 60)
    gx, gy = np.meshgrid(grid, grid)
    grid_design = np.column_stack([np.ones(gx.size), gx.ravel(), gy.ravel(), gx.ravel() ** 2, gy.ravel() ** 2, gx.ravel() * gy.ravel()])
    fields = []
    for idx in range(n_cells):
        eta = grid_design @ coeffs[idx]
        field = (1.0 / (1.0 + np.exp(-np.clip(eta, -20.0, 20.0)))).reshape(gx.shape)
        fields.append(field)
    subset = max(n_cells // 2, 1)
    dec_x_subset = DecodingAlgorithms.linear_decode(spikes[:, :subset], px)
    dec_y_subset = DecodingAlgorithms.linear_decode(spikes[:, :subset], py)
    dec_x_full = DecodingAlgorithms.linear_decode(spikes, px)
    dec_y_full = DecodingAlgorithms.linear_decode(spikes, py)
    return {
        "time_s": time,
        "px": px,
        "py": py,
        "vx": vel[:, 0],
        "vy": vel[:, 1],
        "spikes": spikes,
        "firing_prob": firing_prob,
        "fields": np.asarray(fields, dtype=float),
        "grid_x": gx,
        "grid_y": gy,
        "decoded_subset_x": dec_x_subset["decoded"],
        "decoded_subset_y": dec_y_subset["decoded"],
        "decoded_full_x": dec_x_full["decoded"],
        "decoded_full_y": dec_y_full["decoded"],
        "rmse_full": float(np.sqrt(np.mean((dec_x_full["decoded"] - px) ** 2 + (dec_y_full["decoded"] - py) ** 2))),
    }


def _plot_raster(ax, time_s, spikes, *, max_cells=20):
    n_cells = min(int(spikes.shape[1]), max_cells)
    for row in range(n_cells):
        spike_times = np.asarray(time_s, dtype=float)[np.asarray(spikes[:, row], dtype=float) > 0.5]
        if spike_times.size:
            ax.vlines(spike_times, row + 0.6, row + 1.4, color="k", linewidth=0.35)
    ax.set_ylim(0.5, n_cells + 0.5)
    ax.set_ylabel("cell")


In [ ]:
# SECTION 0: 2-D Stimulus Decode
# This notebook follows the MATLAB 2-D decoding workflow with simulated spatial receptive fields.
plt.close("all")
payload = _simulate_decode()
print({"num_cells": int(payload["spikes"].shape[1]), "rmse_full": round(float(payload["rmse_full"]), 4)})


In [ ]:
# SECTION 1: Generate the random receptive fields to simulate different neurons
fig = _prepare_figure("figure; plot(px,py)", figsize=(6.0, 6.0))
ax = fig.subplots(1, 1)
ax.plot(payload["px"], payload["py"], color="tab:blue", linewidth=1.5)
ax.set_title("Simulated X-Y trajectory")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal", adjustable="box")

fig = _prepare_figure("lambda{i}.plot", figsize=(9.0, 5.0))
ax = fig.subplots(1, 1)
show = [0, 1, 2, 3]
for idx in show:
    ax.plot(payload["time_s"], payload["firing_prob"][:, idx], linewidth=1.2, label=f"Cell {idx + 1}")
ax.set_title("Example firing probabilities")
ax.set_xlabel("time (s)")
ax.set_ylabel("spike probability")
ax.legend(loc="upper right", frameon=False, fontsize=8)

fig = _prepare_figure("pcolor(X,Y,placeField{i}), shading interp", figsize=(8.0, 8.0))
axs = fig.subplots(2, 2, squeeze=False)
for ax, idx in zip(axs.ravel(), show, strict=False):
    image = ax.imshow(
        payload["fields"][idx],
        origin="lower",
        extent=[float(payload["grid_x"].min()), float(payload["grid_x"].max()), float(payload["grid_y"].min()), float(payload["grid_y"].max())],
        aspect="equal",
        cmap="viridis",
    )
    ax.set_title(f"Cell {idx + 1}")
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(image, ax=axs.ravel().tolist(), shrink=0.78)


In [ ]:
# SECTION 2: Visualize the simulated neural activity
fig = _prepare_figure("spikeColl.plot", figsize=(9.0, 5.0))
axs = fig.subplots(2, 1, sharex=True)
_plot_raster(axs[0], payload["time_s"], payload["spikes"])
axs[0].set_title("Population raster")
axs[1].plot(payload["time_s"], np.mean(payload["spikes"], axis=1), color="tab:green", linewidth=1.2)
axs[1].set_title("Population firing fraction")
axs[1].set_xlabel("time (s)")
axs[1].set_ylabel("mean spike/bin")


In [ ]:
# SECTION 3: Decode the x-y trajectory
fig = _prepare_figure("plot(x_u(1,:),x_u(2,:),'b',px,py,'k')", figsize=(6.0, 6.0))
ax = fig.subplots(1, 1)
ax.plot(payload["px"], payload["py"], color="k", linewidth=1.8, label="True path")
ax.plot(payload["decoded_subset_x"], payload["decoded_subset_y"], color="tab:orange", linewidth=1.0, label="Subset decode")
ax.plot(payload["decoded_full_x"], payload["decoded_full_y"], color="tab:blue", linewidth=1.2, label="Full decode")
ax.set_title("Decoded X-Y trajectory")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(loc="best", frameon=False, fontsize=8)
ax.set_aspect("equal", adjustable="box")

fig = _prepare_figure("plot(decoded trajectories)", figsize=(10.0, 5.5))
axs = fig.subplots(2, 1, sharex=True)
axs[0].plot(payload["time_s"], payload["px"], color="k", linewidth=1.6, label="True x")
axs[0].plot(payload["time_s"], payload["decoded_full_x"], color="tab:blue", linewidth=1.2, label="Decoded x")
axs[0].plot(payload["time_s"], payload["decoded_subset_x"], color="tab:orange", linewidth=1.0, label="Subset x")
axs[0].legend(loc="best", frameon=False, fontsize=8)
axs[0].set_ylabel("x")
axs[1].plot(payload["time_s"], payload["py"], color="k", linewidth=1.6, label="True y")
axs[1].plot(payload["time_s"], payload["decoded_full_y"], color="tab:blue", linewidth=1.2, label="Decoded y")
axs[1].plot(payload["time_s"], payload["decoded_subset_y"], color="tab:orange", linewidth=1.0, label="Subset y")
axs[1].set_ylabel("y")
axs[1].set_xlabel("time (s)")

fig = _prepare_figure("decode_rmse", figsize=(7.0, 4.5))
ax = fig.subplots(1, 1)
error_full = np.sqrt((payload["decoded_full_x"] - payload["px"]) ** 2 + (payload["decoded_full_y"] - payload["py"]) ** 2)
error_subset = np.sqrt((payload["decoded_subset_x"] - payload["px"]) ** 2 + (payload["decoded_subset_y"] - payload["py"]) ** 2)
ax.plot(payload["time_s"], error_full, color="tab:blue", linewidth=1.2, label="Full decode")
ax.plot(payload["time_s"], error_subset, color="tab:orange", linewidth=1.0, label="Subset decode")
ax.set_title(f"Pointwise decoding error (RMSE={payload['rmse_full']:.3f})")
ax.set_xlabel("time (s)")
ax.set_ylabel("Euclidean error")
ax.legend(loc="best", frameon=False, fontsize=8)
__tracker.finalize()
